In [ ]:
import numpy as np
import builtins

from mes_packages import *

_print = builtins.print  # sauvegarde
def print(*args, **kwargs):
    def colorize(arg):
        if isinstance(arg, (bool, np.bool_)):
            if arg:  # True
                return f'\033[94m{arg}\033[0m'  # Bleu
            else:    # False
                return f'\033[91m{arg}\033[0m'  # Rouge
        return arg
    
    args = [colorize(arg) for arg in args]
    _print(*args, **kwargs)

builtins.print = print


# Matrice de saut globale 
$$
\sum_{T\in \mathcal{T}} \sum_{F\in\mathcal{F}_T\cap \mathcal{F}_{int}} 
\int_{F}(u_{T}-u_{V})\overline{v_T}
$$
avec $\mathcal{T}$ l'ensemble des éléments et $\mathcal{F}_T\cap \mathcal{F}_{int}$ l'ensemble des faces intérieurs de $T$

In [ ]:
mesh = create_mesh_circle_in_square(radius=0.1, square_size=0.3, mesh_size=0.025)
ordre = 2
MAT_saut = build_jump_matrix_DG(mesh, ordre, verbose=True)


=== Vérification de l'orientation des triangles ===
Nombre total de triangles: 251

Nombre de triangles corrigés: 0/251
Tous les triangles étaient déjà correctement orientés True

=== Assemblage de la matrice de saut globale ===

Nombre de triangles : 251
Nombre de faces intérieures estimé : 753
Nombre d'éléments non nuls estimé : 13554

=== Statistiques d'assemblage ===
Faces intérieures traitées : 678
Faces de bord ignorées : 75
Total de faces : 753

Matrice MAT_saut :
  Taille : 1506 x 1506
  Éléments non nuls utilisés : 12204
  Éléments non nuls alloués : 13554
  Taux de remplissage : 90.04%

Assemblage de la matrice de saut globale terminé


# Construction d'une matrice de saut sr une face

Assemblage par sommation de la matrice :
$$
\int_F(u_T-u_V)\overline{v_T}
$$
On additionne les deux matrices sparses MTT et MTV

In [ ]:
# Préparation d'un maillage 
mesh = create_mesh_circle_in_square(radius=0.1, square_size=0.3, mesh_size=0.025)
ordre = 2
# Recuperation de la géométrie du maillage
points = mesh.points[:, :2]  # On ne garde que les coordonnées x et y
triangles = np.asarray(mesh.cells_dict["triangle"]) # mesh.cells_dict["triangle"]
# Construction de la structure de voisinage
neighbors, neighbor_faces, edges_to_triangles = build_neighborhood_structure(triangles)
loctoglob_DG,Nglob_DG = build_loctoglob_DG(triangles, ordre)
Nloc = (ordre + 1) * (ordre + 2) //2
# Calcul de la matrice de masse de référence 1D
M_ref_1D = build_masse_ref_1D(ordre)

# Formation de la matrice de saut globale
nnz = Nglob_DG * Nloc * 20
MAT = COOMatrix(Nglob_DG, Nglob_DG, nnz)
iT=4
iface=1
MTT = build_masse_frontiere_elt_DG(iT, iface, ordre, loctoglob_DG, points, triangles, M_ref_1D)
MTV = build_boundary_mass_TV(iT, iface, ordre, triangles, points, neighbors, neighbor_faces, M_ref_1D)

# Ajoute à MAT MTT
MAT=MAT + MTT
# Ajoute à MAT MTV
MAT=MAT - MTV


# Test de la forme sesquilinéaire: [f1]^* MAT [f2]


# Création des vecteurs nodaux pour f1(x,y) = x*y et f2(x,y) = y^2
f1 = build_nodal_vector_DG(lambda x, y: x * y, mesh,ordre)
f2 = build_nodal_vector_DG(lambda x, y: y**2, mesh,ordre)


# Calcul : [f1]^* MAT [f2]
result = MAT.sesquilinear_form(f1, f2)

print("\n" + "="*60)
print("=== Test de la forme sesquilinéaire: [f1]^* MAT [f2] ===")
print("="*60)

is_ok = np.isclose(result, 0.0, atol=1e-12)
print(f"valeur nulle: ",is_ok)



=== Vérification de l'orientation des triangles ===
Nombre total de triangles: 251

Nombre de triangles corrigés: 0/251
Tous les triangles étaient déjà correctement orientés True

=== Test de la forme sesquilinéaire: [f1]^* MAT [f2] ===
valeur nulle:  True
